# FinReasoning — standalone evaluation

Run this notebook to evaluate a fine-tuned adapter **with valid metrics**.

**Why Step 4 in the main Colab notebook looked broken:** `data/processed` stores only `prompt`, `completion`, and `task` for SFT. The evaluator needs `answer`, `question`, `context`, and (for numerical tasks) `expression` / `variables`. Loading the test split from disk dropped those fields, so prompts were empty, predictions became *Insufficient information.*, and ground truth appeared empty (NaN in CSV).

**This notebook** rebuilds the **test split from raw JSONL** (same stratified split as preprocessing) and uses the fixed evaluator that builds prompts with `format_as_prompt_completion` (matching training).

**Requirements:** GPU recommended; raw training JSONL under `data/raw/` (or set `RAW_DATA_PATH`); adapter at `outputs/sft_qlora/final_adapter` or set `ADAPTER_DIR`.

**Google Colab:** Run the **Setup** cell below first (mount Drive, clone/pull repo). Paths default to `MyDrive/FinReasoningAI/` like the main training notebook.

## Setup (Google Colab)

Mount Drive and clone or pull **[juankim834/FinReasoningAI](https://github.com/juankim834/FinReasoningAI)** under `MyDrive/FinReasoningAI/FinReasoningAI`. Edit `WORKSPACE` in the next cell if your Drive layout differs.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
WORKSPACE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = os.path.join(WORKSPACE, "FinReasoningAI")

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)

if Path(PROJECT_DIR, ".git").is_dir():
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("Working directory:", os.getcwd())

In [ ]:
import os, sys, gc
from pathlib import Path

ROOT = Path.cwd().resolve()
os.chdir(ROOT)
rp = str(ROOT)
if rp not in sys.path:
    sys.path.insert(0, rp)
print("Working directory:", ROOT)

## Configuration

Defaults: raw JSONL under the Drive workspace `.../FinReasoningAI/data/raw`, adapter under `.../FinReasoningAI/outputs/sft_qlora/final_adapter` (same as the main Colab symlinks). Set `RAW_DATA_PATH` / `ADAPTER_DIR` environment variables to override.

In [ ]:
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-14B-Instruct")

_drive_workspace = ROOT.parent
RAW_DATA_PATH = os.environ.get(
    "RAW_DATA_PATH", str(_drive_workspace / "data" / "raw")
)
ADAPTER_DIR = os.environ.get(
    "ADAPTER_DIR",
    str(_drive_workspace / "outputs" / "sft_qlora" / "final_adapter"),
)

OUTPUT_CSV = "outputs/eval_results.csv"
MAX_SAMPLES = None  # e.g. 50 for a quick smoke test; None = all test samples
MAX_NEW_TOKENS = 128

assert Path(RAW_DATA_PATH).exists(), f"Missing RAW_DATA_PATH: {RAW_DATA_PATH}"
assert Path(ADAPTER_DIR).exists(), f"Missing ADAPTER_DIR: {ADAPTER_DIR}"

print("MODEL_ID:", MODEL_ID)
print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("ADAPTER_DIR:", ADAPTER_DIR)

In [ ]:
from src.data.preprocess import load_eval_test_samples

test_samples = load_eval_test_samples(RAW_DATA_PATH)
print(f"Test split: {len(test_samples)} samples")
if test_samples:
    s0 = test_samples[0]
    print("Example keys:", sorted(s0.keys()))
    print("task:", s0.get("task"))
    print("answer (prefix):", str(s0.get("answer", ""))[:120])

In [ ]:
import torch
from peft import PeftModel
from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# try:
#     import flash_attn  # noqa: F401
#     _attn = "flash_attention_2"
# except ImportError:
#     _attn = "eager"
_attn = "eager"
eval_base, eval_tokenizer = load_model_and_tokenizer(
    MODEL_ID, DEFAULT_BNB_CONFIG, attn_implementation=_attn
)
eval_model = PeftModel.from_pretrained(eval_base, ADAPTER_DIR)
eval_model.eval()

if torch.cuda.is_available():
    print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("Model loaded (CPU — generation will be slow).")

In [ ]:
from src.eval.evaluate import evaluate_model

metrics = evaluate_model(
    model=eval_model,
    tokenizer=eval_tokenizer,
    test_dataset=test_samples,
    output_csv=OUTPUT_CSV,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=MAX_SAMPLES,
)

print("\nEvaluation Results:")
print("-" * 40)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")

In [ ]:
import pandas as pd

results_df = pd.read_csv(OUTPUT_CSV)
# Avoid pandas inferring empty columns as numeric NaN
for col in ("question", "ground_truth", "prediction"):
    if col in results_df.columns:
        results_df[col] = results_df[col].fillna("").astype(str)

print(f"Total evaluated: {len(results_df)} samples\n")
print("Per-task breakdown:")
print(results_df.groupby("task")[["exact_match", "f1", "parsable", "grounding_rate"]].mean().round(4))

bad = results_df[results_df["exact_match"] == 0]
print("\nLow-scoring samples (EM=0), first 15:")
display(bad[["task", "question", "ground_truth", "prediction", "f1", "grounding_rate"]].head(15))